# 04 — Custom Evaluators

**What you'll learn:**
- Writing a **code-based** evaluator (deterministic Python)
- Writing a **prompt-based** evaluator (an LLM judge with your own rubric)
- Mixing custom and built-in evaluators in one `evaluate()` call

---

Built-ins cover general quality and safety, but domain rules are yours to define —
e.g. "does the answer cite a valid policy number?" or "does it stay on-brand?".
Two styles:

1. **Code-based** — any callable that returns a dict of scores. Deterministic, free.
2. **Prompt-based** — a class that asks a judge model to score against your rubric.

In [ ]:
import os
import site
import shutil
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# --------------------------------------------------------------------------
# Reasoning-model compatibility (gpt-5.x / o-series judge models)
# --------------------------------------------------------------------------
# Reasoning models reject `max_tokens` (they require `max_completion_tokens`) and
# only allow the default temperature. azure-ai-evaluation runs evaluators in
# spawned worker processes, so the fix must load in every worker: we copy the
# workshop's `sitecustomize.py` shim into site-packages (auto-imported by every
# Python process) and also apply it here in the kernel. It is a no-op for classic
# models (gpt-4o etc.), so it is always safe to run.
_shim_src = Path("../sitecustomize.py")
if _shim_src.exists():
    _shim_dst = Path(site.getsitepackages()[0]) / "sitecustomize.py"
    if not _shim_dst.exists() or _shim_dst.read_text() != _shim_src.read_text():
        shutil.copyfile(_shim_src, _shim_dst)
    exec(_shim_src.read_text())

# Judge model config for AI-assisted evaluators (notebooks 02, 04, 05).
# Uses az login / DefaultAzureCredential when AZURE_OPENAI_API_KEY is blank.
model_config = {
    "azure_endpoint": os.environ["AZURE_OPENAI_ENDPOINT"],
    "azure_deployment": os.environ["AZURE_OPENAI_DEPLOYMENT"],
    "api_version": os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
}
if os.environ.get("AZURE_OPENAI_API_KEY"):
    model_config["api_key"] = os.environ["AZURE_OPENAI_API_KEY"]

# Foundry project (notebooks 03, 06) — enables cloud logging + safety evaluators.
azure_ai_project = os.environ.get("AZURE_AI_PROJECT_ENDPOINT")

DATA_DIR = "../data"
print("Judge deployment:", model_config["azure_deployment"])

## 1. A code-based evaluator

An evaluator is just a callable. It receives the row's fields as keyword arguments
and returns a dict. Here we check whether a response mentions a **claim-filing deadline**
in the expected numeric form — a simple business rule.

In [ ]:
import re


class DeadlineMentionEvaluator:
    """Checks the response states a filing deadline in '<N> days' form."""

    def __call__(self, *, response: str, **kwargs) -> dict: # type: ignore
        match = re.search(r"(\d+)\s*days", response.lower())
        mentions_deadline = match is not None
        return {
            "mentions_deadline": mentions_deadline,
            "deadline_days": int(match.group(1)) if match else -1,
        } # type: ignore


deadline_eval = DeadlineMentionEvaluator()
deadline_eval(response="You must file within 30 days of the accident.")

## 2. A prompt-based (LLM-judge) evaluator

For judgment calls, let a judge model score against your own rubric. A prompt-based
evaluator is just a class that calls the judge and returns a score dict — the same
`model_config` you used for the built-ins. Here we score whether an insurance answer
is written in **plain, jargon-free language** (1–5).

> We call the Azure OpenAI client directly so this works with any judge deployment,
> including reasoning models (the setup cell's compatibility shim handles those).

In [ ]:
import json
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider


def _judge_client(cfg: dict) -> AzureOpenAI:
    kwargs = dict(
        azure_endpoint=cfg["azure_endpoint"],
        api_version=cfg.get("api_version", "2024-10-21"),
    )
    if cfg.get("api_key"):
        kwargs["api_key"] = cfg["api_key"]
    else:  # Entra ID via az login
        kwargs["azure_ad_token_provider"] = get_bearer_token_provider(
            DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
        )
    return AzureOpenAI(**kwargs)


class PlainLanguageEvaluator:
    """LLM-judge: scores 1 (dense jargon) to 5 (clear plain English)."""

    def __init__(self, model_config: dict):
        self._client = _judge_client(model_config)
        self._deployment = model_config["azure_deployment"]

    def __call__(self, *, query: str, response: str, **kwargs) -> dict:
        prompt = (
            "You are a plain-language reviewer for insurance customer support. "
            "Score the RESPONSE from 1 (dense jargon) to 5 (clear, plain English a "
            "layperson understands). Return JSON: "
            '{"plain_language": <int>, "reason": "<short reason>"}.\n\n'
            f"QUERY: {query}\nRESPONSE: {response}"
        )
        completion = self._client.chat.completions.create(
            model=self._deployment,
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            max_tokens=200,
        )
        return json.loads(completion.choices[0].message.content)


plain_language_eval = PlainLanguageEvaluator(model_config)
plain_language_eval(
    query="What is a deductible?",
    response="A deductible is what you pay before your coverage kicks in.",
)

## 3. Combine them in one run

`evaluate()` happily mixes custom and built-in evaluators. Use `column_mapping`
if an evaluator expects a differently-named field (see notebook 05).

In [ ]:
from azure.ai.evaluation import evaluate, F1ScoreEvaluator

result = evaluate(
    data=f"{DATA_DIR}/insurance-qa.jsonl",
    evaluators={
        "deadline": DeadlineMentionEvaluator(),
        "plain_language": plain_language_eval,
        "f1": F1ScoreEvaluator(),
    },
)
result["metrics"]

## Recap

- Code-based evaluators are plain callables returning a score dict — deterministic and free.
- Prompt-based evaluators use a judge model to score against your own rubric.
- Custom and built-in evaluators run together in a single `evaluate()`.

**Next:** [05 — Evaluating a target](./05-evaluating-a-target.ipynb) — score a live app,
not just a static dataset.